# Day 39: Semantic Routing for Agentic AI

Welcome to Day 39 of the AI Engineering Mastery program! Today, we explore how to implement a **Semantic Router** to classify user intent and route queries to specialized agents.

## Core Theory: The "Why" and "How"

As we transition into building multi-agent systems, treating every user query the exact same way is inefficient. 

**Why Semantic Routing?**
- **Efficiency:** Not every query requires an expensive, multi-step ReAct (Reasoning and Acting) loop. Simple queries can be answered quickly, while complex analytical questions are routed to heavy-duty data agents.
- **Specialization:** Agents perform best when they have a narrow, well-defined scope (specific prompts and specific tools). Routing allows you to split the problem space into distinct domains.
- **Security/Control:** You can route out-of-domain or potentially harmful queries to a safety fallback agent before they touch your core business logic.

**How does it work?**
A Semantic Router sits at the very entry point of your architecture. Instead of rigid keyword matching or regex, it leverages LLMs (via structured output/function calling) or Embedding comparisons to dynamically classify the semantic intent of the input text. 

Today, we will implement an LLM-based router using LangChain and Pydantic. By forcing the LLM to output a strict JSON schema, we guarantee that the output perfectly matches our predefined routing categories.

## Common Pitfalls in Production
1. **Overlapping Intents:** If your categories are too similar (e.g., `sales_agent` vs `pricing_agent`), the router will struggle and lower its confidence. Keep intents mutually exclusive and distinct.
2. **Latency Overhead:** Using a massive, slow model (like GPT-4) just to route a query adds unnecessary latency. Use faster, smaller models (like GPT-3.5-Turbo or Claude 3 Haiku) specifically tuned for the routing step.
3. **Missing Fallbacks:** Users *will* ask things your system wasn't designed for. Always have a fallback mechanism for low-confidence scores or unrecognized intents.

## Code Implementation: Building the Router

Below is a production-grade implementation of a Semantic Router using LangChain, LCEL (LangChain Expression Language), and Pydantic for structured intent classification.

In [1]:
import os
from enum import Enum
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

class RouteIntent(str, Enum):
    """Supported routing destinations for our system."""
    CUSTOMER_SUPPORT = "customer_support"
    DATA_ANALYTICS = "data_analytics"
    TECHNICAL_DOCS = "technical_docs"
    GENERAL_CHAT = "general_chat"

class RouterOutput(BaseModel):
    """Structured output schema enforced on the LLM."""
    intent: RouteIntent = Field(
        description="The primary intent of the user query."
    )
    confidence_score: float = Field(
        description="Confidence in the classification between 0.0 and 1.0."
    )
    reasoning: str = Field(
        description="Brief explanation for why this intent was chosen."
    )

def build_semantic_router(model_name: str = "gpt-3.5-turbo"):
    """
    Builds a LangChain LCEL chain that classifies a query and outputs structured routing data.
    
    Args:
        model_name: The OpenAI model to use for intent classification.
        
    Returns:
        A LangChain Runnable that yields a RouterOutput instance.
    """
    # Initialize the LLM with temperature 0 for deterministic routing
    llm = ChatOpenAI(model=model_name, temperature=0.0)
    
    # Bind the Pydantic model to force structured JSON output
    structured_llm = llm.with_structured_output(RouterOutput)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert intent classification router. "
                   "Analyze the user's query and classify it into one of the exact specified intents. "
                   "If the query doesn't strongly fit analytics, support, or technical docs, default to general_chat."),
        ("user", "Query: {query}")
    ])
    
    # LCEL pipeline: Prompt -> Structured LLM
    return prompt | structured_llm

# ------------------------------------------------------------------
# Mock Destination Agents
# ------------------------------------------------------------------

def support_agent(query: str) -> str:
    """Destination agent for customer support queries."""
    return f"[Support Agent] Handling ticket for: '{query}'"

def analytics_agent(query: str) -> str:
    """Destination agent for data analytics queries."""
    return f"[Analytics Agent] Executing SQL/Pandas for: '{query}'"

def general_agent(query: str) -> str:
    """Destination agent for general conversational queries."""
    return f"[General Agent] Chitchat response for: '{query}'"

def execute_routing(router_output: RouterOutput, query: str) -> str:
    """
    Dynamically invokes the correct agent based on the classified intent.
    
    Args:
        router_output: The parsed intent object from the LLM.
        query: The original user query.
        
    Returns:
        The final response string from the destination agent.
    """
    print(f"-> Routed to: {router_output.intent.name} (Confidence: {router_output.confidence_score:.2f})")
    print(f"-> Reasoning: {router_output.reasoning}\n")
    
    if router_output.intent == RouteIntent.CUSTOMER_SUPPORT:
        return support_agent(query)
    elif router_output.intent == RouteIntent.DATA_ANALYTICS:
        return analytics_agent(query)
    else:
        return general_agent(query)

def run_demo():
    """Executes a demonstration of the Semantic Router pipeline."""
    router_chain = build_semantic_router()
    
    test_queries = [
        "I forgot my password and my account is locked! Help!",
        "Show me the month-over-month revenue growth for the European sector.",
        "What is the capital of France?"
    ]
    
    for q in test_queries:
        print(f"=== Processing Query: '{q}' ===")
        try:
            # Execute the router chain
            result: RouterOutput = router_chain.invoke({"query": q})
            
            # Pass the result to our execution function
            final_response = execute_routing(result, q)
            print(f"Result: {final_response}\n")
        except Exception as e:
            # Graceful fallback for local execution without valid API keys
            print(f"[Bypassed API Execution] Error: {e}\n")

if __name__ == "__main__":
    run_demo()


/app/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:2575: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


=== Processing Query: 'I forgot my password and my account is locked! Help!' ===


[Bypassed API Execution] Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

=== Processing Query: 'Show me the month-over-month revenue growth for the European sector.' ===
[Bypassed API Execution] Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

=== Processing Query: 'What is the capital of France?' ===
[Bypassed API Execution] Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status'

## Practical Lab / Homework

**Your Task:** 
The current router defaults to `GENERAL_CHAT` if it doesn't know what to do, but it blindly trusts the LLM regardless of the `confidence_score`.

1. Modify the `RouteIntent` Enum to include a `FALLBACK` intent.
2. Update the `execute_routing` function to inspect the `confidence_score`.
3. If the confidence score drops below **0.75**, explicitly override the routed intent and send the query to a new `fallback_agent(query)` function.
4. Test it with a highly ambiguous or nonsensical query to trigger the fallback condition.

In [2]:
# Write your Lab Implementation here.

